# 🏥 Fine-Tuning Expérimental d'un Modèle Médical (Hackathon TechCorp)

Ce notebook permet de fine-tuner un modèle d'intelligence artificielle avec le dataset médical `ruslanmv/ai-medical-chatbot`. Il est conçu pour s'exécuter sur **Google Colab** (avec GPU T4 gratuit ou Pro).

**Spécialité : IA - L'Expert Modèles**

## 1. Installation des dépendances

In [ ]:
!pip install -q transformers peft datasets bitsandbytes trl accelerate

## 2. Importation des bibliothèques

In [ ]:
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset

## 3. Configuration du modèle avec optimisation mémoire (Quantization 4-bit)

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

print(f"🤖 Chargement du modèle de base : {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=False,
    low_cpu_mem_usage=True,
)

if len(tokenizer) > model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Préparation du Dataset

In [ ]:
print("📂 Téléchargement du dataset : ruslanmv/ai-medical-chatbot")
dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")

# On prend un sous-ensemble pour accélérer l'expérience si désiré (dé-commentez pour un test rapide)
# dataset = dataset.select(range(2000))

def format_conversation(example):
    patient = example.get('Patient', example.get('Description', ''))
    doctor = example.get('Doctor', example.get('Doctor', ''))
    text = f"<|user|>\n{patient}<|end|>\n<|assistant|>\n{doctor}<|end|>"
    return {"text": text}

formatted_dataset = dataset.map(format_conversation)

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names
)
print("✅ Dataset formaté et tokenisé.")

## 5. Lancement de l'entraînement

In [ ]:
output_dir = "./medical_model_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1, # 1 époque pour du prototypage rapide
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    max_steps=50,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    dataloader_drop_last=True
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("⏳ Lancement de l'entraînement LoRA...")
trainer.train()

print(f"✅ Sauvegarde du modèle dans {output_dir}")
trainer.save_model()